<a href="https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/08_react_practice/notebook_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8: ReAct Practice

This notebook explores practical the ReAct (Reasoning and Acting) pattern with Google's Gemini API. We will use the `google-genai` library to interact with Gemini models. It includes a mock search tool, a thought generation phase using structured outputs, and an action phase with function calling, all orchestrated by a ReAct control loop.

**Learning Objectives:**

1. Understand how ReAct breaks problems into Thought → Action → Observation.
2. Practice orchestrating the full ReAct loop end-to-end.

> **Exercise version.** This is the exercise notebook for Lesson 8. The full solution lives in [`notebook.ipynb`](https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/08_react_practice/notebook.ipynb) in the same folder. Attempt each exercise before checking the solutions.

### Exercise roadmap

| Exercise | Difficulty | What you build |
|---|---|---|
| 1 | Starter | The XML tool description the agent reasons over |
| 2 | Intermediate | The Thought phase: plain-text reasoning about the next step |
| 3 | Advanced | The Action phase: tool call or final answer via function calling |
| 4 | Advanced | The full Thought, Action, Observation control loop |

## 1. Setup


### Set Up Python Environment

**Google Colab:** Run the code cell below — it installs all required packages and automatically loads your credentials from Colab Secrets (your `GOOGLE_API_KEY`, or your Vertex AI settings if you chose that option in the Course Admin lesson).

To set up your Python virtual environment using `uv` and load it into the Notebook, follow the step-by-step instructions from the `Course Admin` lesson from the beginning of the course.

**TL;DR:** Be sure the correct kernel pointing to your `uv` virtual environment is selected.

In [ ]:
# ============================================================
# Google Colab Setup — runs only when executed in Colab
# ============================================================
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Install the course package (published from pyproject.toml) and its pinned extras
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "agentic-ai-engineering-course==0.4.17",
            "nest-asyncio2",
            "google-auth==2.53.0",
            "opentelemetry-api==1.42.1",
            "opentelemetry-sdk==1.42.1",
            "opentelemetry-exporter-otlp-proto-http==1.42.1",
            "opentelemetry-exporter-otlp-proto-common==1.42.1",
            "opentelemetry-proto==1.42.1",
            "jedi==0.18.2",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without restart

In [ ]:
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

from utils import env

env.load(required_env_vars=["GOOGLE_API_KEY"])

### Import Key Packages

In [ ]:
from enum import Enum
from typing import Callable

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from utils import pretty_print

### Initialize the Gemini Client

In [ ]:
client = genai.Client()

### Define Constants

We will use the `gemini-3.7-flash` model, which is fast and cost-effective:

In [ ]:
MODEL_ID = "gemini-3.7-flash"

## 2. Tools Definition

Let's implement our mock search tool that will serve as the external knowledge source for our agent. This simplified version focuses on the ReAct mechanics rather than real API integration:

In [ ]:
def search(query: str) -> str:
    """Search for information about a specific topic or query.

    Args:
        query (str): The search query or topic to look up.
    """
    query_lower = query.lower()

    # Predefined responses for demonstration
    if all(word in query_lower for word in ["capital", "france"]):
        return "Paris is the capital of France and is known for the Eiffel Tower."
    elif "react" in query_lower:
        return "The ReAct (Reasoning and Acting) framework enables LLMs to solve complex tasks by interleaving thought generation, action execution, and observation processing."

    # Generic response for unhandled queries
    return f"Information about '{query}' was not found."

We maintain a mapping from tool name to tool function (the tool registry). This lets the model plan with symbolic tool names, while our code safely resolves those names to actual Python functions to execute.

In [ ]:
TOOL_REGISTRY: dict[str, Callable[..., str]] = {
    search.__name__: search,
}

## 3. ReAct Thought Phase

Now let's implement the thought generation phase. This component analyzes the current situation and determines what the agent should do next, potentially suggesting using tools.

First, we prepare the prompt for the thinking part. We implement a function that converts the `TOOL_REGISTRY` to a string XML representation of it, which we insert into the prompt. This way, the LLM knows which tools available and can reason around them.

### Exercise 1: Build the tool descriptions for the prompt

The Thought phase reasons about which tools could help. For that, the prompt needs a readable description of every registered tool, generated straight from the functions' docstrings.

**Learning goal:** Generate a prompt-ready XML description of a tool registry.

**What you need to implement:**

1. Loop over the registry's (name, function) pairs
2. For each tool, emit an opening tag carrying the tool's name, its docstring lines wrapped in a description tag (skip the description block if the docstring is empty), and a closing tag
3. Join all lines with newlines and return the result

**Key concepts:**

- `fn.__doc__` returns a function's docstring (possibly `None`, hence the `or ""` guard pattern)
- Tab characters (`\t`) indent the XML so it reads nicely inside the prompt

**Expected output:** the solved cell below prints the thought prompt with a `<tool name="search">` block inside `<tools>`, including the search function's docstring.

**Implementation hints:**

- Build a list of lines and join once at the end, string concatenation in a loop is the messier path
- Until this returns real content, the printed prompt below just shows an empty tools section

In [ ]:
# === Exercise cell: fill in the gaps below ===


def build_tools_xml_description(tool_registry: dict[str, Callable[..., str]]) -> str:
    """Build a minimal XML description of tools using only their docstrings.

    Steps to complete:
    1. Loop through the registry's (tool_name, fn) pairs
    2. For each tool: append an opening tag with the tool's name, then the
       docstring lines wrapped in a description tag (skip the description
       block when the docstring is empty), then a closing tag
    3. Join the collected lines with newlines and return the result
    """
    lines = []

    # Your implementation goes here

    return "\n".join(lines)


# Build a string of XML describing the tools
tools_xml = build_tools_xml_description(TOOL_REGISTRY)

PROMPT_TEMPLATE_THOUGHT = """
You are deciding the next best step for reaching the user goal. You have some tools available to you.

Available tools:
<tools>
{tools_xml}
</tools>

Conversation so far:
<conversation>
{conversation}
</conversation>

State your next **thought** about what to do next as one short paragraph focused on the next action you intend to take and why.
Avoid repeating the same strategies that didn't work previously. Prefer different approaches.

Remember:
- Return ONLY plain natural language text.
- Do NOT emit JSON, XML, function calls, or code.
""".strip()

### Validation check - run this after your implementation

Uncomment the cell below and run it once Exercise 1 is done. It checks the generated XML locally, no API calls.

In [ ]:
# # Validation: check your Exercise 1 implementation
# try:
#     assert tools_xml, "❌ tools_xml is empty. Implement the function and re-run its cell."
#     assert '<tool name="search">' in tools_xml, "❌ Expected a tool tag carrying the name 'search'."
#     assert "<description>" in tools_xml and "</description>" in tools_xml, "❌ Expected the docstring wrapped in description tags."
#     assert "Search for information" in tools_xml, "❌ The search docstring text should appear in the description."
#     assert "</tool>" in tools_xml, "❌ Every tool block needs a closing tag."
#     print("✅ All checks passed! The agent can now read its tools.")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: collect lines into a list and remember the closing tag for each tool.")

Here we `print` the prompt with the tool definitions inside.

In [ ]:
print(PROMPT_TEMPLATE_THOUGHT.format(tools_xml=tools_xml, conversation=""))

We can now implement the `generate_thought` function, which reasons on the best next action to take according to the conversation history.

### Exercise 2: Generate the Thought

The Thought phase asks the model, in plain text, what it should do next given the conversation so far and the available tools. No JSON, no function calls, just reasoning.

**Learning goal:** Implement the reasoning step of the ReAct cycle.

**What you need to implement:**

1. Build the tools XML from the registry (your Exercise 1 function)
2. Fill `PROMPT_TEMPLATE_THOUGHT` with the tools XML and the conversation
3. Call the model with the formatted prompt
4. Return the response text, stripped of surrounding whitespace

**Key concepts:**

- The template's placeholders are `tools_xml` and `conversation`, filled via `str.format`
- `client.models.generate_content()` with just model and contents is a plain text call, structured output is deliberately not used here

**Expected output:** the smoke test prints one short reasoning paragraph, typically concluding that the search tool should be used for the France question.

**Implementation hints:**

- This function is called fresh every turn, the growing conversation string is what makes each thought different
- The smoke test makes one cheap API call once implemented, none while the placeholder is in place

In [ ]:
# === Exercise cell: fill in the gaps below ===


def generate_thought(conversation: str, tool_registry: dict[str, Callable[..., str]]) -> str:
    """Generate a thought as plain text (no structured output).

    Steps to complete:
    1. Build the tools XML description from the registry
    2. Format PROMPT_TEMPLATE_THOUGHT with the tools XML and the conversation
    3. Call the model with the formatted prompt
    4. Return the stripped response text
    """
    # Your implementation goes here

    return ""  # Replace with the model's thought text


# Try it out (one API call once implemented)
thought = generate_thought("User: What is the capital of France?", TOOL_REGISTRY)
print(thought if thought else "(empty, complete the function above)")

## 4. ReAct Action Phase

Next, let's implement the action phase using function calling. This component determines whether to use a tool or provide a final answer.

### Exercise 3: Generate the Action

After thinking, the agent commits to an action: either request a tool call or deliver a final answer. This is where native function calling enters the loop, and where the `force_final` escape hatch lives.

**Learning goal:** Implement the decision step that returns either a `ToolCallRequest` or a `FinalAnswer`.

**What you need to implement:**

1. Forced branch: when `force_final` is true or no tools are provided, format `PROMPT_TEMPLATE_ACTION_FORCED`, call the model without tools, and return the text as a `FinalAnswer`
2. Normal branch: format `PROMPT_TEMPLATE_ACTION`, build a config that passes the registry's functions as tools with automatic function calling disabled, and call the model
3. Scan the response candidate's content parts: the first part carrying a function call becomes a `ToolCallRequest` with the call's name and arguments
4. If no part contains a function call, join the parts' text into a `FinalAnswer`

**Key concepts:**

- `types.GenerateContentConfig` accepts plain Python functions via `tools`, the SDK builds their schemas
- `types.AutomaticFunctionCallingConfig(disable=True)` stops the SDK from running tools itself, the loop wants to execute them explicitly
- `part.function_call` carries `.name` and `.args` (convert args to a plain dict)
- `ToolCallRequest` and `FinalAnswer` are the two Pydantic shapes defined above your gap

**Expected output:** the smoke test prints a `ToolCallRequest` for the France question, showing the search tool's name and a query argument.

**Implementation hints:**

- Handle the forced branch first and return early, it keeps the normal branch flat
- `dict(part.function_call.args or {})` guards against calls with no arguments

In [ ]:
# === Exercise cell: fill in the gaps below ===

PROMPT_TEMPLATE_ACTION = """
You are selecting the best next action to reach the user goal.

Conversation so far:
<conversation>
{conversation}
</conversation>

Respond either with a tool call (with arguments) or a final answer, but only if you can confidently conclude.
""".strip()

# Dedicated prompt used when we must force a final answer
PROMPT_TEMPLATE_ACTION_FORCED = """
You must now provide a final answer to the user.

Conversation so far:
<conversation>
{conversation}
</conversation>

Provide a concise final answer that best addresses the user's goal.
""".strip()


class ToolCallRequest(BaseModel):
    """A request to call a tool with its name and arguments."""

    tool_name: str = Field(description="The name of the tool to call.")
    arguments: dict = Field(description="The arguments to pass to the tool.")


class FinalAnswer(BaseModel):
    """A final answer to present to the user when no further action is needed."""

    text: str = Field(description="The final answer text to present to the user.")


def generate_action(
    conversation: str, tool_registry: dict[str, Callable[..., str]] | None = None, force_final: bool = False
) -> ToolCallRequest | FinalAnswer:
    """Generate an action by passing tools to the LLM and parsing function calls or final text.

    When force_final is True or no tools are provided, the model is instructed to produce a final answer
    and tool calls are disabled.

    Steps to complete:
    1. Forced branch (force_final is True or no tools): format the forced
       template, call the model without tools, return a FinalAnswer with the
       stripped text
    2. Normal branch: format the action template, build a config that passes
       the registry's functions as tools and disables automatic function
       calling, then call the model with it
    3. Scan the first candidate's content parts: if a part carries a function
       call, return a ToolCallRequest built from its name and arguments
    4. Otherwise join the parts' text and return it as a stripped FinalAnswer
    """
    # Your implementation goes here

    return FinalAnswer(text="")  # Replace with the parsed action


# Try it out (one API call once implemented)
action = generate_action("User: What is the capital of France?", TOOL_REGISTRY)
if isinstance(action, FinalAnswer) and not action.text:
    print("(placeholder result, complete the function above)")
else:
    print(action)

Why we provide an option to force the final answer? In a ReAct loop we sometimes need to terminate cleanly after a budget of turns (e.g., to avoid infinite loops or excessive tool calls). The force flag lets us ask the model to conclude with a final answer even if, under normal conditions, it might keep calling tools. This ensures graceful shutdown and a usable output at the end of the loop.

Note: In the Action phase we do not inline tool descriptions into the prompt (unlike the Thought phase). Instead, we pass the available Python tool functions through the `tools` parameter to `generate_content`. The client automatically parses these tools and incorporates their definitions/arguments into the model's prompt context, enabling function calling without duplicating tool specs in our prompt text.

## 5. ReAct Observation Phase

This is the third main component of the ReAct loop. In this step, we take the ToolCallRequest created by the generate_action function, run the tool, and return the output.

In [ ]:
def observe(action_request: ToolCallRequest, tool_registry: dict[str, Callable[..., str]]) -> str:
    """
    Execute the selected tool and return the observation text
    (either a result or an error message)
    """
    name = action_request.tool_name
    args = action_request.arguments

    if name not in tool_registry:
        return f"Unknown tool '{name}'. Available: {', '.join(tool_registry)}"

    try:
        return tool_registry[name](**args)
    except Exception as e:
        return f"Error executing tool '{name}': {e}"

In [ ]:
req = ToolCallRequest(tool_name="search", arguments={"query": "capital of France"})
print(observe(req, TOOL_REGISTRY))

## 6. ReAct Control Loop

Now we build the main ReAct control loop that orchestrates the Thought → Action → Observation cycle end-to-end. We treat the conversation between the user and the agent as a sequence of messages. Each message is a step in the dialogue, and each step corresponds to one ReAct unit: it can be a user message, an internal thought, a tool request, the tool's observation, or the final answer.

We'll start by defining the data structures for these messages.

In [ ]:
class MessageRole(str, Enum):
    """Enumeration for the different roles a message can have."""

    USER = "user"
    THOUGHT = "thought"
    TOOL_REQUEST = "tool request"
    OBSERVATION = "observation"
    FINAL_ANSWER = "final answer"


class Message(BaseModel):
    """A message with a role and content, used for all message types."""

    role: MessageRole = Field(description="The role of the message in the ReAct loop.")
    content: str = Field(description="The textual content of the message.")

    def __str__(self) -> str:
        """Provides a user-friendly string representation of the message."""
        return f"{self.role.value.capitalize()}: {self.content}"

We also add a small printer that uses our `pretty_print` module to render each message nicely in the notebook. This makes it easy to follow how the agent alternates between Thought, Action (tool call), and Observation across turns.

In [ ]:
def pretty_print_message(
    message: Message,
    turn: int,
    max_turns: int,
    header_color: str = pretty_print.Color.YELLOW,
    is_forced_final_answer: bool = False,
) -> None:
    if not is_forced_final_answer:
        title = f"{message.role.value.capitalize()} (Turn {turn}/{max_turns}):"
    else:
        title = f"{message.role.value.capitalize()} (Forced):"

    pretty_print.wrapped(
        text=message.content,
        title=title,
        header_color=header_color,
    )

We now use a `Scratchpad` class that wraps a list of `Message` objects and provides `append(..., verbose=False)` to both store and (optionally) pretty-print messages with role-based colors. The scratchpad is serialized each turn so the model can plan the next step.

In [ ]:
class Scratchpad:
    """Container for ReAct messages with optional pretty-print on append."""

    def __init__(self, max_turns: int) -> None:
        self.messages: list[Message] = []
        self.max_turns: int = max_turns
        self.current_turn: int = 1

    def set_turn(self, turn: int) -> None:
        self.current_turn = turn

    def append(self, message: Message, verbose: bool = False, is_forced_final_answer: bool = False) -> None:
        self.messages.append(message)
        if verbose:
            role_to_color = {
                MessageRole.USER: pretty_print.Color.RESET,
                MessageRole.THOUGHT: pretty_print.Color.ORANGE,
                MessageRole.TOOL_REQUEST: pretty_print.Color.GREEN,
                MessageRole.OBSERVATION: pretty_print.Color.YELLOW,
                MessageRole.FINAL_ANSWER: pretty_print.Color.CYAN,
            }
            header_color = role_to_color.get(message.role, pretty_print.Color.YELLOW)
            pretty_print_message(
                message=message,
                turn=self.current_turn,
                max_turns=self.max_turns,
                header_color=header_color,
                is_forced_final_answer=is_forced_final_answer,
            )

    def to_string(self) -> str:
        return "\n".join(str(m) for m in self.messages)

We can now implement the control loop.
- On the first turn, we add the user question.
- Then, at each turn: (1) we get a Thought from the model; (2) we get an Action. If the action is a `FinalAnswer`, we stop. If it's a `ToolCallRequest`, we execute the tool and append the resulting `Observation`, then continue. If we reach the maximum number of turns, we run the action selector one last time with a flag that forces a final answer (no tool calls).

### Exercise 4: Orchestrate the ReAct control loop

Every piece exists: thoughts, actions, observations, the `Scratchpad`, and the `Message` types. The loop is where they become an agent.

**Learning goal:** Compose Thought, Action, and Observation into the full ReAct cycle with turn limits and a forced finish.

**What you need to implement:**

1. Create a `Scratchpad` and append the user's question as a `Message` with the user role
2. For each turn from 1 to `max_turns`: update the scratchpad's turn, generate a thought from the serialized scratchpad and append it
3. Generate an action from the serialized scratchpad: a `FinalAnswer` gets appended and its text returned, a `ToolCallRequest` gets logged (format its arguments), executed via `observe`, and its observation appended
4. On the last turn, call the action generator once more with the flag that forces a final answer, append the result (marking it as forced), and return the text, falling back to an apology string if the model still refuses

**Key concepts:**

- `scratchpad.to_string()` serializes the whole history, it is the context for both thought and action generation
- `scratchpad.append(message, verbose=verbose)` handles the color-coded printing for you
- `isinstance(action_result, FinalAnswer)` versus `ToolCallRequest` drives the branch
- The forced-final append accepts `is_forced_final_answer=True` for distinct display

**Expected output:** running the test cells below with `verbose=True` shows the full colored trace: User, Thought, Tool request, Observation per turn, ending in a Final answer.

**Implementation hints:**

- The turn-limit check happens inside the loop after handling the action, comparing the current turn against `max_turns`
- Format tool request log lines as `tool_name(param=value, ...)` so the trace reads naturally
- The test cells below print nothing until this is implemented, then each run costs a handful of API calls (2 per turn plus the forced finish)

In [ ]:
# === Exercise cell: fill in the gaps below ===


def react_agent_loop(
    initial_question: str, tool_registry: dict[str, Callable[..., str]], max_turns: int = 5, verbose: bool = False
) -> str | None:
    """
    Implements the main ReAct (Thought -> Action -> Observation) control loop.
    Uses a unified message class for the scratchpad.

    Steps to complete:
    1. Create a Scratchpad and append the user's question
    2. Loop over the turns: set the scratchpad turn, generate a thought from
       the serialized scratchpad, and append it
    3. Generate an action from the serialized scratchpad:
       - FinalAnswer: append it and return its text
       - ToolCallRequest: append a tool-request message, run observe(), and
         append the observation
    4. When the last turn is reached, generate one more action with the
       force-final flag, append it as a forced final answer, and return its
       text (fall back to an apology string if it is not a FinalAnswer)
    """
    # Your implementation goes here

    return None  # Replace: return the agent's final answer text

Let's test our ReAct agent with a simple factual question that requires a search:

In [ ]:
# A straightforward question requiring a search.
question = "What is the capital of France?"
final_answer = react_agent_loop(question, TOOL_REGISTRY, max_turns=2, verbose=True)

Last, let's test it with a question that our mock search tool doesn't have knowledge about:

In [ ]:
# A question about a concept the mock search tool doesn't know.
question = "What is the capital of Italy?"
final_answer = react_agent_loop(question, TOOL_REGISTRY, max_turns=2, verbose=True)

### Validation check - run this after your implementation

Uncomment the cell below and run it after the two test cells above have produced a trace. It inspects the last run's result, no extra API calls.

In [ ]:
# # Validation: check your Exercise 4 implementation
# try:
#     assert final_answer is not None, "❌ final_answer is None. Implement the loop and re-run the test cells above."
#     assert isinstance(final_answer, str) and final_answer.strip(), "❌ The loop should return a non-empty answer string."
#     print("✅ All checks passed! Your ReAct loop runs end-to-end.")
#     print(f"Last answer: {final_answer[:120]}")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: make sure both the early FinalAnswer branch and the forced last-turn branch return the answer text.")

Notice how the ReAct agent tried different strategies to find an answer for the user query, demonstrating live adaptation.

## 7. ReAct with a reasoning model (use built‑in “thinking”)

Here, we set a thinking level, and define two helpers. The first helper extracts any thought summary text the API returns. 

The second finds the first function call in a response when the model decides to use a tool.

In [ ]:
THINKING_CONFIG = types.ThinkingConfig(
    include_thoughts=True,  # human-readable summaries for transparency/debugging
    thinking_level="low",  # Gemini 3.x reasoning depth: minimal | low | medium | high
)


def extract_thought_summary(response: types.GenerateContentResponse) -> str | None:
    """Collect human-readable thought summaries if present."""
    parts = getattr(response.candidates[0].content, "parts", []) or []
    chunks = [p.text for p in parts if getattr(p, "thought", False) and getattr(p, "text", None)]
    return "\n".join(chunks).strip() if chunks else None


def extract_first_function_call(response: types.GenerateContentResponse):
    """Return (name, args) for the first function call, or None if the model produced a final answer."""
    if getattr(response, "function_calls", None):
        fc = response.function_calls[0]
        return fc.name, dict(fc.args or {})
    parts = getattr(response.candidates[0].content, "parts", []) or []
    for p in parts:
        if getattr(p, "function_call", None):
            return p.function_call.name, dict(p.function_call.args or {})
    return None

Here, we build the request configuration. We provide the Python functions as tools and enable built-in thinking. Automatic function calling is disabled, so we can log each step and run tools ourselves with the `observe` function. 

In [ ]:
def build_config_with_tools(tools: list[Callable[..., str]]) -> types.GenerateContentConfig:
    return types.GenerateContentConfig(
        tools=tools,
        thinking_config=THINKING_CONFIG,
        # We disable the automatic execution of tools, we will use the observe function to run them instead.
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )

The following is the alternative ReAct loop. The conversation is maintained as a list of `types.Content`. 

After each model turn, we append `response.candidates[0].content` back into `contents` to preserve thought signatures. 

When the model calls a tool, we execute it, log the observation, and then append a `function_response` part so the model can use that observation on the next turn. 

For the visible trace, we keep using a `Scratchpad` object (which we call `human_log`) and our `pretty_print_message` helper function.

In [ ]:
def react_agent_loop_thinking(
    initial_question: str,
    tool_registry: dict[str, Callable[..., str]],
    max_turns: int = 5,
    verbose: bool = True,
) -> str:
    """
    ReAct loop that relies on model-native reasoning:
      - optional thought summaries for visibility,
      - thought signatures preserved by appending model Content back into `contents`,
      - pretty-printed trace using Lesson 8's Scratchpad utilities.
    """

    human_log = Scratchpad(max_turns=max_turns)
    human_log.append(Message(role=MessageRole.USER, content=initial_question), verbose=verbose)

    # Structured "contents" conversation for thought signatures
    contents: list[types.Content] = [types.Content(role="user", parts=[types.Part(text=initial_question)])]
    tools = list(tool_registry.values())
    config = build_config_with_tools(tools)

    for turn in range(1, max_turns + 1):
        human_log.set_turn(turn)

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=contents,
            config=config,
        )

        # 1) Thought summary (if any) — log as your THOUGHT message
        thoughts = extract_thought_summary(response)
        if thoughts:
            human_log.append(Message(role=MessageRole.THOUGHT, content=thoughts), verbose=verbose)

        # 2) Function/Tool call?
        fc = extract_first_function_call(response)
        if fc:
            name, args = fc

            # We keep the model's full response content to preserve the thought signatures
            contents.append(response.candidates[0].content)

            # Log the tool request
            params_str = ", ".join(f"{k}={repr(v)}" for k, v in args.items())
            human_log.append(
                Message(role=MessageRole.TOOL_REQUEST, content=f"{name}({params_str})"),
                verbose=verbose,
            )

            # Execute the tool
            action_request = ToolCallRequest(tool_name=name, arguments=args)
            observation = observe(action_request, tool_registry)

            # Log observation
            human_log.append(Message(role=MessageRole.OBSERVATION, content=observation), verbose=verbose)

            # Send the function response back (standard function-calling protocol)
            fn_resp = types.Part.from_function_response(
                name=name,
                response={"result": observation},
            )
            contents.append(types.Content(role="user", parts=[fn_resp]))
            continue  # next turn

        # 3) No function call => final text
        final_text = (response.text or "").strip()
        human_log.append(Message(role=MessageRole.FINAL_ANSWER, content=final_text), verbose=verbose)
        return final_text

    # 4) Forced finish if we hit max turns: disable tool-calling for the last shot
    forced_config = types.GenerateContentConfig(
        thinking_config=THINKING_CONFIG,
        tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode=types.FunctionCallingConfigMode.NONE)),
    )
    forced_response = client.models.generate_content(model=MODEL_ID, contents=contents, config=forced_config)
    final_text = (forced_response.text or "Unable to produce a final answer within the allotted turns.").strip()
    human_log.append(
        Message(role=MessageRole.FINAL_ANSWER, content=final_text),
        verbose=verbose,
        is_forced_final_answer=True,
    )
    return final_text

Now let’s test this new loop using the same questions we used earlier. 

In [ ]:
question = "What is the capital of France?"
final_answer = react_agent_loop_thinking(question, TOOL_REGISTRY, max_turns=3, verbose=True)

We get the same answer, but now the Thought comes from the summarized version of the model’s internal thinking. Notice how “verbose” these thought summaries are by default. 

Now let’s ask our agent the second question.

In [ ]:
question = "What is the capital of Italy?"
final_answer = react_agent_loop_thinking(question, TOOL_REGISTRY, max_turns=3, verbose=True)

## Stretch challenges

Want to go further? Try these on your own:

1. Add a reflection step after each observation: a second thought that judges whether the tool result actually helped, appended to the scratchpad with a new role.
2. Add a `max_failures` counter that forces a final answer after two consecutive observations containing "not found" or errors, instead of burning all remaining turns.
3. Register a second tool (for example a calculator) and ask a question that needs both tools, then compare how the classic loop and the thinking-model loop from section 7 sequence their calls.